# 🎯 YOLOv11x-cls - Plant Disease Classification

**Model:** YOLOv11 Extra-Large Classification

**Why YOLOv11x:**
- Latest YOLO (2024 release)
- Extra-large version (more parameters than nano)
- Classification mode (not detection)

**Configuration:**
- 640x640 images
- 10 epochs with early stopping
- Batch size: 16 (larger model)

**Expected:** Better than YOLOv8/v11n baselines

---

## ⚠️ Enable GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q ultralytics opendatasets
print("✅ Installed")

In [ ]:
import os, opendatasets as od
if not os.path.exists('/content/plantvillage'):
    od.download("https://www.kaggle.com/datasets/mohitsingh1804/plantvillage")
if not os.path.exists('/content/PlantDoc-Dataset'):
    !git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
print("✅ Datasets ready")

In [ ]:
import shutil, time, json
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

pv_base = Path('/content/plantvillage/PlantVillage') if Path('/content/plantvillage/PlantVillage').exists() else Path('/content/plantvillage')
PV_TRAIN, PV_VAL, PD_TEST = pv_base/'train', pv_base/'val', Path('/content/PlantDoc-Dataset/test')

In [ ]:
common_labels = sorted([
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry_(including_sour)___healthy", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Grape___healthy", "Peach___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Raspberry___healthy", "Soybean___healthy",
    "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight",
    "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites_Two-spotted_spider_mite",
    "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
])

plantdoc_to_plantvillage = {
    "Apple leaf": "Apple___healthy", "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Apple Scab Leaf": "Apple___Apple_scab", "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot", "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry_(including_sour)___healthy", "Corn Gray leaf spot": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight", "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Peach leaf": "Peach___healthy", "Potato leaf": "Potato___healthy",
    "Potato leaf early blight": "Potato___Early_blight", "Potato leaf late blight": "Potato___Late_blight",
    "Raspberry leaf": "Raspberry___healthy", "Soybean leaf": "Soybean___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew", "Strawberry leaf": "Strawberry___healthy",
    "Tomato Early blight leaf": "Tomato___Early_blight", "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy", "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight", "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites_Two-spotted_spider_mite",
    "grape leaf": "Grape___healthy", "grape leaf black rot": "Apple___Black_rot"
}

print(f"Classes: {len(common_labels)}")

In [ ]:
# Prepare YOLO dataset
def prepare_yolo_dataset(pv_train, pv_val, pd_test, output_dir='/content/yolo_data'):
    out = Path(output_dir)
    for split in ['train', 'val', 'test']:
        (out / split).mkdir(parents=True, exist_ok=True)
    
    print("📂 Preparing YOLO dataset...\n")
    
    print("Copying training data...")
    for label in tqdm(common_labels):
        src, dst = Path(pv_train) / label, out / 'train' / label
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    
    print("Copying validation data...")
    for label in tqdm(common_labels):
        src, dst = Path(pv_val) / label, out / 'val' / label
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    
    print("Copying test data...")
    for pd_label, pv_label in tqdm(plantdoc_to_plantvillage.items()):
        if pv_label in common_labels:
            src, dst = Path(pd_test) / pd_label, out / 'test' / pv_label
            dst.mkdir(parents=True, exist_ok=True)
            if src.exists():
                for ext in ['*.jpg', '*.JPG', '*.png', '*.PNG']:
                    for img_file in src.glob(ext):
                        shutil.copy(img_file, dst)
    
    print(f"\n✅ YOLO dataset ready: {out}")
    return str(out)

yolo_dataset_path = prepare_yolo_dataset(str(PV_TRAIN), str(PV_VAL), str(PD_TEST))

In [ ]:
# Train YOLOv11x-cls
print("\n" + "="*60)
print("Training YOLOv11x-cls (Extra-Large)")
print("="*60 + "\n")

t0 = time.time()

# Load YOLOv11x classification model
model = YOLO('yolov11x-cls.pt')

# Train
results = model.train(
    data=yolo_dataset_path,
    epochs=10,
    imgsz=640,
    batch=16,  # Smaller batch for larger model
    patience=5,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    project='/content/yolo_runs',
    name='yolov11x-cls',
    verbose=True
)

print("\n✅ Training complete!")

In [ ]:
# Validate on test set
print("\nValidating on PlantDoc test set...\n")

val_results = model.val(split='test')
training_time = time.time() - t0

# Extract metrics
test_acc_top1 = val_results.top1 * 100 if hasattr(val_results, 'top1') else 0
test_acc_top5 = val_results.top5 * 100 if hasattr(val_results, 'top5') else 0

print(f"\n{'='*60}")
print("FINAL RESULTS - YOLOv11x-cls")
print(f"{'='*60}")
print(f"Test Accuracy (Top-1): {test_acc_top1:.2f}%")
print(f"Test Accuracy (Top-5): {test_acc_top5:.2f}%")
print(f"Training Time: {training_time/60:.1f} minutes")
print(f"{'='*60}")

In [ ]:
# Save results
results_dict = {
    'model': 'YOLOv11x-cls (Extra-Large)',
    'image_size': 640,
    'batch_size': 16,
    'epochs': 10,
    'test_acc_top1': test_acc_top1,
    'test_acc_top5': test_acc_top5,
    'training_time_minutes': training_time/60
}

with open('/content/yolov11x_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Top-1 Acc', 'Top-5 Acc']
values = [test_acc_top1, test_acc_top5]
colors = ['#FF6B6B', '#4ECDC4']
ax.bar(metrics, values, color=colors, alpha=0.8)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('YOLOv11x-cls Test Performance (PlantDoc)', fontweight='bold')
ax.set_ylim([0, 100])
for i, v in enumerate(values):
    ax.text(i, v+2, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('/content/yolov11x_results.png', dpi=300)
plt.show()

print("\n✅ Saved: yolov11x_results.json, yolov11x_results.png")
print("\n💡 Model checkpoint: /content/yolo_runs/yolov11x-cls/weights/best.pt")

In [ ]:
from google.colab import files
files.download('/content/yolov11x_results.json')
files.download('/content/yolov11x_results.png')
print("✅ Downloaded!")